# Prompt Engineering Techniques: Basic Techniques

Source slide: `slides/prompt-engineering/04.1_Prompt_Engineering_Techniques.Basic_Techniques.md`

Each example here matches a basic technique explicitly named in the slide.



## Prerequisites

- Install the real GitHub Copilot SDK: `pip install github-copilot-sdk`
- Sign in to GitHub Copilot in VS Code or GitHub Codespaces
- These examples use ambient auth; do not paste a PAT into the notebook

Each technique below has a real Copilot SDK failure run, a short failure test, and an improved run that shows the fix.


In [ ]:
from copilot import CopilotClient
from copilot.session import PermissionHandler

model = "gpt-4.1"

async def ask_copilot(
    prompt: str,
    *,
    system_message: str | None = None,
    model_name: str = model,
) -> str:
    """Run a single prompt through the real GitHub Copilot SDK using ambient auth."""
    async with CopilotClient() as client:
        session_kwargs = {
            "model": model_name,
            "on_permission_request": PermissionHandler.approve_all,
        }
        if system_message:
            session_kwargs["system_message"] = {
                "mode": "append",
                "content": system_message,
            }
        async with await client.create_session(**session_kwargs) as session:
            response = await session.send_and_wait(prompt)
            return response.data.content if response and response.data else ""

def show_block(title: str, content: str) -> None:
    print(title)
    print("=" * 80)
    print(content.strip())
    print()

async def run_prompt_pair(
    *,
    technique_name: str,
    failure_prompt: str,
    improved_prompt: str,
    failure_test: str,
    fix_explanation: str,
    system_message: str | None = None,
    config_note: str | None = None,
) -> None:
    show_block("📘 Technique", technique_name)
    if config_note:
        show_block("⚙️ Configuration note", config_note)
    show_block("❌ Failure prompt", failure_prompt)
    failure_result = await ask_copilot(
        failure_prompt,
        system_message=system_message,
    )
    show_block("❌ Failure result", failure_result)
    show_block("🧪 Failure test", failure_test)
    show_block("✅ Improved prompt", improved_prompt)
    improved_result = await ask_copilot(
        improved_prompt,
        system_message=system_message,
    )
    show_block("✅ Improved result", improved_result)
    show_block("🔧 Why this fixes it", fix_explanation)

print("✅ Setup complete - real GitHub Copilot SDK with ambient auth")
print(f"📦 Using model: {model}")


## Clear and Specific Instructions

**Failure mode:** A vague prompt forces the model to choose its own scope, structure, and success criteria.

**Failure test:** The failure prompt uses a broad question with no format guidance, so the answer can sprawl instead of solving the exact user need.

**Technique:** Tell the model exactly what to cover, how much to write, and what structure to follow.

**Improved example:** The improved prompt specifies three bullets and one sentence per bullet, so the model has much less room to wander.


In [ ]:
await run_prompt_pair(
    technique_name='Clear and Specific Instructions',
    failure_prompt='Explain climate change.',
    improved_prompt='Summarize the primary causes of climate change in exactly three bullet points. Keep each bullet to one sentence.',
    failure_test='The failure prompt uses a broad question with no format guidance, so the answer can sprawl instead of solving the exact user need.',
    fix_explanation='The improved prompt specifies three bullets and one sentence per bullet, so the model has much less room to wander.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Reference Text and Citations

**Failure mode:** Without reference text, the model answers from memory and cannot prove where its claims came from.

**Failure test:** The failure prompt asks a policy question without providing the policy, so the answer may sound correct while remaining unverifiable.

**Technique:** Include the relevant source text and require the answer to cite it.

**Improved example:** The improved prompt embeds the policy excerpt and asks for a citation, which grounds the answer and makes review easier.


In [ ]:
await run_prompt_pair(
    technique_name='Reference Text and Citations',
    failure_prompt='What does our data retention policy say about deleting inactive accounts?',
    improved_prompt='Using only the policy excerpt below, answer the question and cite the exact bullet you used.\n\nPolicy excerpt:\n- Delete inactive trial accounts after 30 days.\n- Delete inactive paid accounts after 180 days.\n\nQuestion: What does our data retention policy say about deleting inactive accounts?',
    failure_test='The failure prompt asks a policy question without providing the policy, so the answer may sound correct while remaining unverifiable.',
    fix_explanation='The improved prompt embeds the policy excerpt and asks for a citation, which grounds the answer and makes review easier.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Task Decomposition

**Failure mode:** Complex work often fails when too many operations are implied but not separated.

**Failure test:** The failure prompt requests analysis and a recommendation in one breath, so the model may skip the evidence-gathering step and jump to advice.

**Technique:** Break the task into explicit ordered sub-steps.

**Improved example:** The improved prompt tells the model to extract facts first, summarize second, and recommend third, which mirrors the reasoning workflow you want.


In [ ]:
await run_prompt_pair(
    technique_name='Task Decomposition',
    failure_prompt='Analyze this table and tell me what we should do next. Q1 churn: 7%, Q2 churn: 9%, Q3 churn: 12%.',
    improved_prompt='Work in three numbered steps.\n1. Extract the churn values from this data: Q1 churn: 7%, Q2 churn: 9%, Q3 churn: 12%.\n2. Summarize the trend in one sentence.\n3. Suggest one actionable next step.',
    failure_test='The failure prompt requests analysis and a recommendation in one breath, so the model may skip the evidence-gathering step and jump to advice.',
    fix_explanation='The improved prompt tells the model to extract facts first, summarize second, and recommend third, which mirrors the reasoning workflow you want.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Instruction Placement

**Failure mode:** Important instructions are easier to miss when they are buried after long context blocks.

**Failure test:** The failure prompt places the citation requirement after a long context section, so the model may focus on the content and ignore the formatting instruction.

**Technique:** Put the main task and critical output requirements in a prominent location.

**Improved example:** The improved prompt moves the answer rule next to the task, so the model sees the requirement before it processes the supporting context.


In [ ]:
await run_prompt_pair(
    technique_name='Instruction Placement',
    failure_prompt='Context: Our pricing notes say annual customers save 20%, enterprise customers get SSO, and monthly customers can cancel anytime. After reading the context above, summarize the main benefits. Also include one citation.',
    improved_prompt='Summarize the main benefits in exactly two bullets and include one quoted phrase as evidence.\n\nContext: Annual customers save 20%, enterprise customers get SSO, and monthly customers can cancel anytime.',
    failure_test='The failure prompt places the citation requirement after a long context section, so the model may focus on the content and ignore the formatting instruction.',
    fix_explanation='The improved prompt moves the answer rule next to the task, so the model sees the requirement before it processes the supporting context.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)


## Output Priming and Syntax

**Failure mode:** Even when the model understands the task, it may return the answer in a shape that is awkward to reuse.

**Failure test:** The failure prompt requests a list, but it does not prime the syntax, so the answer can come back as prose paragraphs.

**Technique:** Show the exact format or syntax you want the model to emit.

**Improved example:** The improved prompt includes the bullet template, which primes the structure before generation starts.


In [ ]:
await run_prompt_pair(
    technique_name='Output Priming and Syntax',
    failure_prompt='List three types of renewable energy.',
    improved_prompt='List three types of renewable energy. Answer in this exact format:\n- item 1\n- item 2\n- item 3',
    failure_test='The failure prompt requests a list, but it does not prime the syntax, so the answer can come back as prose paragraphs.',
    fix_explanation='The improved prompt includes the bullet template, which primes the structure before generation starts.',
    system_message='You are a precise prompt-engineering teaching assistant. Follow the prompt exactly.',
)
